# PhysAR-IDS -- NB9 (v6): Cross-dataset validation on the MSU / Morris power-system dataset

**Why this notebook exists.** The dissertation's earlier MSU numbers came from a leakage-era pipeline whose code is gone. This notebook rebuilds the whole cross-dataset study from `msu_combined.csv`, with the same discipline as the v6 Sweeten runs: temporal splits, 5 seeds `[42, 7, 123, 2024, 99]`, mean +/- std, paired t-tests with Holm correction.

**The data.** 78,377 rows, 128 features: four PMU relays R1-R4 x 29 columns (12 voltage phasors, 12 current phasors, frequency, frequency change, impedance, impedance angle, status flag) plus 12 log columns (control panel, relay and Snort logs). 37 scenario markers (`marker`): `41` = no event (normal), `1-6, 13, 14` = natural events (8), all other markers = attacks (28). The file is 15 consecutive passes over the 37 scenarios, so every (pass, marker) is one contiguous time series (~123 rows). **No timestamp exists; row order inside a run is taken as time order.**

**Tasks.** 3-class (normal / natural / attack) and 37-class (marker). Models: XGBoost (default), XGBoost (tuned), Random Forest, MLP, Logistic Regression. "XGBoost (default)" means library defaults (100 trees, depth 6, lr 0.3) **plus subsample = colsample_bytree = 0.8**; without subsampling XGBoost is deterministic and all 5 seeds would give identical models, which makes every paired test over seeds degenerate.

**Three evaluation splits (this is the important part).**
* **A - temporal within run:** first 70% of each run trains, 5-row gap, remaining rows test. Same logic as the Sweeten split. Train and test come from the *same recordings*.
* **B - held-out passes:** passes 1-10 train, passes 11-15 test. Test runs are separate recordings of each scenario, so this is the honest generalisation test.
* **C - random 70/30 (leakage control):** the kind of split the old numbers used. Only for XGBoost, to measure how much a random split inflates the score.

**What is and is not tested.** Tested: architecture ranking, effect of tuning, feature-group ablation (PMU vs logs vs derived), Physical-Channel Reliance on this dataset, per-class difficulty, split inflation. **Not tested here:** zero-shot transfer from the Sweeten model (no shared features: MSU has no Modbus/TCP fields and Sweeten has no phasor angles, so there is nothing to transfer) and the adversarial-training effect (needs a perturbation model for PMU data that the old pipeline never defined; that would be NB9b).

**Units.** All rates are fractions in [0, 1] and printed with 4 decimals, so 1.58% is 0.0158 everywhere (the old text mixed the two).

## 0. Setup

In [ ]:
import os, time, json, warnings, platform
import numpy as np, pandas as pd, joblib
import matplotlib; matplotlib.use('Agg') if os.environ.get('NB9_HEADLESS') else None
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import loguniform, randint, uniform
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import ParameterSampler, train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
import xgboost as xgb
warnings.filterwarnings('ignore')

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
CSV_PATH = r'C:\Users\user\Desktop\smartrid1\msu_combined.csv'     # <- edit if the file lives elsewhere
if not os.path.exists(CSV_PATH):
    for alt in ['msu_combined.csv', r'C:\Users\user\Desktop\smartrid1\SmartGrid_Dissertation\msu_combined.csv']:
        if os.path.exists(alt): CSV_PATH = alt; break
assert os.path.exists(CSV_PATH), f'msu_combined.csv not found: set CSV_PATH ({CSV_PATH})'
OUT_DIR = os.path.dirname(os.path.abspath(CSV_PATH)); os.chdir(OUT_DIR)
N_TUNE_TRIALS = 20
SHAP_ROWS = 1500
N_JOBS = -1
print('xgboost', xgb.__version__, '| out dir:', os.getcwd(), '| seeds', SEEDS)

## 1. Load and clean

* `Rk-PA:Z` (impedance) is `inf` in 10,906 rows (a zero-current division in the relay). Each `inf` is replaced by that column's largest finite value and a 0/1 flag column `Rk_Z_inf` records it, so no information is lost and trees/MLP both work.
* `Rk:S` (status flag) is kept numeric.
* `Rk:F == 0` rows are relay dropouts; kept as they are (the model sees them), counted below.

**Derived physical features (per relay, 4 each = 16)**, the MSU analogue of the Sweeten derived features:
* `voltage_imbalance = (max - min) / mean` over the three phase voltage magnitudes `PM1-3:V`
* `current_imbalance = (max - min) / mean` over the three phase current magnitudes `PM4-6:I`
* `power_factor_proxy = cos(angle(V1) - angle(I1))`, with `PA1:VH` and `PA4:IH` in degrees (an angle-based proxy, not a measured power factor)
* `freq_deviation = F - 60` Hz

In [ ]:
raw = pd.read_csv(CSV_PATH)
assert raw.shape[1] == 129 and 'marker' in raw.columns
marker = raw['marker'].astype(int).values
RELAYS = ['R1', 'R2', 'R3', 'R4']
LOG_COLS = [c for c in raw.columns if c != 'marker' and not c.startswith('R')]
PMU_COLS = [c for c in raw.columns if c.startswith('R')]
print('PMU columns', len(PMU_COLS), '| log columns', len(LOG_COLS), LOG_COLS)

X = raw.drop(columns='marker').astype('float64').copy()
FLAG_COLS = []
for r in RELAYS:
    z = f'{r}-PA:Z'; bad = ~np.isfinite(X[z].values)
    X[z] = np.where(bad, X.loc[~bad, z].max(), X[z]); X[f'{r}_Z_inf'] = bad.astype(float); FLAG_COLS.append(f'{r}_Z_inf')
    print(f'{r}: inf impedance rows {bad.mean():.4f} | F==0 rows {(X[f"{r}:F"] == 0).mean():.4f}')
DER_COLS = []
for r in RELAYS:
    V = X[[f'{r}-PM{i}:V' for i in (1, 2, 3)]].values; I = X[[f'{r}-PM{i}:I' for i in (4, 5, 6)]].values
    X[f'{r}_voltage_imbalance'] = (V.max(1) - V.min(1)) / np.maximum(V.mean(1), 1e-9)
    X[f'{r}_current_imbalance'] = (I.max(1) - I.min(1)) / np.maximum(I.mean(1), 1e-9)
    X[f'{r}_power_factor_proxy'] = np.cos(np.deg2rad(X[f'{r}-PA1:VH'] - X[f'{r}-PA4:IH']))
    X[f'{r}_freq_deviation'] = X[f'{r}:F'] - 60.0
    DER_COLS += [f'{r}_voltage_imbalance', f'{r}_current_imbalance', f'{r}_power_factor_proxy', f'{r}_freq_deviation']
assert np.isfinite(X.values).all()
FEATS = list(X.columns); Xv = X.values.astype('float32')
GROUPS = {'pmu_raw': PMU_COLS, 'derived': DER_COLS, 'flags': FLAG_COLS, 'logs': LOG_COLS}
assert sum(len(v) for v in GROUPS.values()) == len(FEATS) == 128 + 16 + 4
print({k: len(v) for k, v in GROUPS.items()}, '| total features', len(FEATS))

## 2. Labels, runs, and the marker-direction check

The old text had `normal_marker = 1` (wrong). Here the direction is **checked from the data** before anything is trained: a no-event scenario should have the smallest log activity and the least PMU disturbance of all 37 scenarios.

In [ ]:
NATURAL = {1, 2, 3, 4, 5, 6, 13, 14}; NORMAL = {41}
ATTACK = set(np.unique(marker)) - NATURAL - NORMAL
assert len(ATTACK) == 28 and len(NATURAL) == 8
g3 = np.where(np.isin(marker, list(NORMAL)), 0, np.where(np.isin(marker, list(NATURAL)), 1, 2))
G3_NAMES = ['normal', 'natural', 'attack']
MARKERS = np.sort(np.unique(marker)); m2i = {m: i for i, m in enumerate(MARKERS)}; y37 = np.array([m2i[m] for m in marker])

# runs = contiguous blocks of one marker; pass index = occurrence number of that marker
start = np.r_[True, marker[1:] != marker[:-1]]; run_id = np.cumsum(start) - 1
n_runs = run_id.max() + 1
run_len = np.bincount(run_id); pos = np.arange(len(marker)) - np.flatnonzero(start)[run_id]
run_marker = marker[start]; pass_of_run = np.zeros(n_runs, int); seen = {}
for r_, m_ in enumerate(run_marker): pass_of_run[r_] = seen.get(m_, 0); seen[m_] = pass_of_run[r_] + 1
pass_id = pass_of_run[run_id]
print(f'{n_runs} runs, run length min/median/max = {run_len.min()}/{int(np.median(run_len))}/{run_len.max()}, passes = {pass_id.max()+1}')

# direction check
logact = (X[LOG_COLS].values != 0).mean(1)
dist = np.abs(X[[f'{r}-PM{i}:V' for r in RELAYS for i in (1, 2, 3)]].values - np.median(X[[f'{r}-PM{i}:V' for r in RELAYS for i in (1, 2, 3)]].values, 0)).mean(1)
chk = pd.DataFrame({'marker': marker, 'log_activity': logact, 'volt_dev': dist, 'status_nonzero': (X[[f'{r}:S' for r in RELAYS]].values != 0).mean(1)}).groupby('marker').mean()
chk['group'] = [G3_NAMES[g3[marker == m_][0]] for m_ in chk.index]
print(chk.sort_values('log_activity').round(4).head(8).to_string())
print('\nmean by group:'); print(chk.groupby('group')[['log_activity', 'volt_dev', 'status_nonzero']].mean().round(4).to_string())
print('\nrows per group:', dict(zip(G3_NAMES, np.bincount(g3))), '| rows per 3-class split is below')

## 3. Splits

`SPLITS[name] = (train_idx, test_idx)`. Split A and B are deterministic (the only randomness across seeds is model randomness, so the seed std is *model* variance, not data-sampling variance; the dissertation should say so). Split C draws a new random split per seed.

In [ ]:
def split_A(frac=0.70, gap=5):
    cut = (run_len * frac).astype(int)[run_id]
    return np.flatnonzero(pos < cut), np.flatnonzero(pos >= cut + gap)
def split_B(n_train_passes=10):
    return np.flatnonzero(pass_id < n_train_passes), np.flatnonzero(pass_id >= n_train_passes)
def split_C(seed):
    return train_test_split(np.arange(len(marker)), test_size=0.30, stratify=y37, random_state=seed)
def tune_split():       # inside split-A train: fit on first 56% of a run, validate on the 5-row-gapped next 14%
    c1 = (run_len * 0.56).astype(int)[run_id]; c2 = (run_len * 0.70).astype(int)[run_id]
    return np.flatnonzero(pos < c1), np.flatnonzero((pos >= c1 + 5) & (pos < c2))
SPLITS = {'A_temporal': split_A(), 'B_heldout_passes': split_B()}
for k, (a, b) in SPLITS.items():
    print(f'{k:18s} train {len(a):6d} test {len(b):6d} | test class counts {np.bincount(g3[b], minlength=3)} | markers in test {len(np.unique(marker[b]))} / train {len(np.unique(marker[a]))}')
    assert len(np.intersect1d(a, b)) == 0

## 4. Models and evaluation

In [ ]:
def fit_predict(name, Xtr, ytr, Xte, seed, cols=None, params=None, return_model=False):
    cols = slice(None) if cols is None else cols
    Xtr, Xte = Xtr[:, cols], Xte[:, cols]
    cls = np.unique(ytr); remap = {c: i for i, c in enumerate(cls)}; yt = np.array([remap[c] for c in ytr])
    if name.startswith('xgb'):
        p = dict(n_estimators=100, max_depth=6, learning_rate=0.3, tree_method='hist', n_jobs=N_JOBS, random_state=seed, verbosity=0)
        if params: p.update(params); p['random_state'] = seed
        if name == 'xgb_default': p.update(subsample=0.8, colsample_bytree=0.8)   # XGBoost defaults except row/column subsampling, so seeds give different models
        m = xgb.XGBClassifier(**p).fit(Xtr, yt); pred = m.predict(Xte)
    elif name == 'rf':
        m = RandomForestClassifier(n_estimators=300, n_jobs=N_JOBS, random_state=seed).fit(Xtr, yt); pred = m.predict(Xte)
    else:
        sc = RobustScaler().fit(Xtr); f = lambda A: np.clip(sc.transform(A), -10, 10)
        if name == 'mlp':
            m = MLPClassifier((128, 64), early_stopping=True, max_iter=80, random_state=seed)
        else:
            m = LogisticRegression(max_iter=300, random_state=seed)
        m.fit(f(Xtr), yt); pred = m.predict(f(Xte))
    pred = cls[pred]
    return (pred, m) if return_model else pred

def score(y_true, y_pred, task, markers_te):
    # group-level attack-detection metrics derived from predictions in either task
    if task == '37class':
        gp = np.array([0 if MARKERS[p] in NORMAL else 1 if MARKERS[p] in NATURAL else 2 for p in y_pred])
        gt = np.array([0 if MARKERS[p] in NORMAL else 1 if MARKERS[p] in NATURAL else 2 for p in y_true])
    else: gp, gt = y_pred, y_true
    atk, pa = gt == 2, gp == 2
    return dict(macro_f1=f1_score(y_true, y_pred, average='macro', labels=np.unique(y_true)), acc=accuracy_score(y_true, y_pred),
                group_macro_f1=f1_score(gt, gp, average='macro', labels=np.unique(gt)),
                attack_recall=float(pa[atk].mean()) if atk.any() else np.nan, attack_fpr=float(pa[~atk].mean()) if (~atk).any() else np.nan)
TASKS = {'3class': g3, '37class': y37}

## 5. Tuning XGBoost (once, seed 42, on a time-blocked validation inside split-A train)

20 random-search trials on the 3-class task (fast); the same parameters are applied to both tasks and both splits. Nothing from any test set is used.

In [ ]:
space = dict(n_estimators=randint(100, 401), max_depth=randint(3, 11), learning_rate=loguniform(0.02, 0.3), subsample=uniform(0.5, 0.5),
             colsample_bytree=uniform(0.3, 0.7), min_child_weight=randint(1, 51), reg_lambda=loguniform(0.1, 10))
tr_i, va_i = tune_split(); t0 = time.time(); trials = []
for k, p in enumerate(ParameterSampler(space, N_TUNE_TRIALS, random_state=42)):
    pr = fit_predict('xgb_tuned', Xv[tr_i], g3[tr_i], Xv[va_i], 42, params=p)
    f = f1_score(g3[va_i], pr, average='macro'); trials.append((f, p)); print(f'trial {k:2d}  val macro-F1 {f:.4f}  ({time.time()-t0:.0f}s)')
best_f, BEST = max(trials, key=lambda t: t[0])
BEST = {k: (int(v) if k in ('n_estimators', 'max_depth', 'min_child_weight') else float(v)) for k, v in BEST.items()}
default_val = f1_score(g3[va_i], fit_predict('xgb_default', Xv[tr_i], g3[tr_i], Xv[va_i], 42), average='macro')
print('\nbest', round(best_f, 4), BEST, '| default on same validation', round(default_val, 4))
json.dump({'best_val_macro_f1': best_f, 'default_val_macro_f1': default_val, 'params': BEST}, open(f'msu_xgb_tuned_params_{PIPELINE_VERSION}.json', 'w'), indent=1)

## 6. Main experiment: 5 seeds x 2 splits x 2 tasks x 5 models

Splits A and B for all models. Expect the MLP and Logistic Regression to be the slow ones.

In [ ]:
MODELS = ['xgb_default', 'xgb_tuned', 'rf', 'mlp', 'lr']
rows = []; CONF = {}; PERCLS = {}; t0 = time.time()
for s in SEEDS:
    for sp, (a, b) in SPLITS.items():
        for task, y in TASKS.items():
            for mn in MODELS:
                pr = fit_predict(mn, Xv[a], y[a], Xv[b], s, params=BEST if mn == 'xgb_tuned' else None)
                rows.append(dict(seed=s, split=sp, task=task, model=mn, **score(y[b], pr, task, marker[b])))
                if mn == 'xgb_tuned':
                    if task == '3class': CONF[(s, sp)] = confusion_matrix(y[b], pr, labels=[0, 1, 2])
                    else: PERCLS[(s, sp)] = f1_score(y[b], pr, average=None, labels=np.arange(len(MARKERS)))
    print(f'seed {s} done ({time.time()-t0:.0f}s)')
R = pd.DataFrame(rows)
ms = lambda x: f'{np.mean(x):.4f} +/- {np.std(x, ddof=1):.4f}'
summ = R.groupby(['split', 'task', 'model']).agg(**{c: (c, ms) for c in ['macro_f1', 'acc', 'attack_recall', 'attack_fpr']}).reset_index()
print(summ.to_string(index=False))

## 7. Statistics: architecture ranking and the effect of tuning

Paired t-test over the 5 seeds (same seeds for every model), Holm-corrected within each (split, task) family. Because splits A and B are fixed, the test only sees model-initialisation variance; read the p-values as "is the gap larger than seed noise", not as a population claim.

In [ ]:
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0
    for rank, i in enumerate(o): run = max(run, (len(p) - rank) * p[i]); adj[i] = min(1.0, run)
    return adj
pair_rows = []
for (sp, task), d in R.groupby(['split', 'task']):
    piv = d.pivot(index='seed', columns='model', values='macro_f1').loc[SEEDS]
    best = piv.mean().idxmax(); fam = []
    for other in [m for m in MODELS if m != best]:
        t, p = stats.ttest_rel(piv[best], piv[other]); fam.append((other, (piv[best] - piv[other]).mean(), p))
    adj = holm([f[2] for f in fam])
    for (o_, dm, p), pa in zip(fam, adj): pair_rows.append(dict(split=sp, task=task, best=best, vs=o_, mean_diff=dm, p_raw=p, p_holm=pa))
    t, p = stats.ttest_rel(piv['xgb_tuned'], piv['xgb_default'])
    pair_rows.append(dict(split=sp, task=task, best='tuned-vs-default', vs='xgb_default', mean_diff=(piv['xgb_tuned'] - piv['xgb_default']).mean(), p_raw=p, p_holm=np.nan))
PAIR = pd.DataFrame(pair_rows); print(PAIR.round(4).to_string(index=False))
rank = R.groupby(['split', 'task', 'model']).macro_f1.mean().groupby(['split', 'task']).rank(ascending=False).unstack('model')
print('\nrank of each model by mean macro-F1 (1 = best):'); print(rank.to_string())

## 8. Leakage control: how much does a random split inflate the score?

XGBoost (default parameters) on a stratified random 70/30 split (split C), against the same model on A and B. This quantifies how much of the old MSU result was split artefact.

In [ ]:
leak = []
for s in SEEDS:
    a, b = split_C(s)
    for task, y in TASKS.items():
        pr = fit_predict('xgb_default', Xv[a], y[a], Xv[b], s); leak.append(dict(seed=s, split='C_random', task=task, model='xgb_default', **score(y[b], pr, task, marker[b])))
L = pd.concat([pd.DataFrame(leak), R[R.model == 'xgb_default']])
leak_tab = L.groupby(['task', 'split']).macro_f1.agg(['mean', 'std']).round(4); print(leak_tab.to_string())
for task in TASKS:
    c = L[(L.task == task) & (L.split == 'C_random')].set_index('seed').macro_f1.loc[SEEDS]
    for sp in SPLITS:
        o = L[(L.task == task) & (L.split == sp)].set_index('seed').macro_f1.loc[SEEDS]
        print(f'{task:8s} random minus {sp:18s}: {np.mean(c - o):+.4f}  (paired p = {stats.ttest_rel(c, o)[1]:.2g})')

## 9. Feature-group ablation

XGBoost (default), 3-class and 37-class, on A and B: all features / PMU raw only / PMU + derived (no logs, no flags) / logs only / without derived features. This replaces the missing ablation in the old write-up.

In [ ]:
ix = {c: i for i, c in enumerate(FEATS)}; idx = lambda *g: [ix[c] for k in g for c in GROUPS[k]]
ABL = {'all': idx('pmu_raw', 'derived', 'flags', 'logs'), 'pmu_raw_only': idx('pmu_raw'), 'pmu_plus_derived': idx('pmu_raw', 'derived'),
       'logs_only': idx('logs'), 'no_derived': idx('pmu_raw', 'flags', 'logs'), 'no_logs': idx('pmu_raw', 'derived', 'flags')}
ab = []
for s in SEEDS:
    for sp, (a, b) in SPLITS.items():
        for task, y in TASKS.items():
            for an, cols in ABL.items():
                pr = fit_predict('xgb_default', Xv[a], y[a], Xv[b], s, cols=cols); ab.append(dict(seed=s, split=sp, task=task, ablation=an, **score(y[b], pr, task, marker[b])))
AB = pd.DataFrame(ab)
AB_t = AB.groupby(['split', 'task', 'ablation']).macro_f1.agg(['mean', 'std']).round(4).reset_index(); print(AB_t.to_string(index=False))
ab_p = []
for (sp, task), d in AB.groupby(['split', 'task']):
    piv = d.pivot(index='seed', columns='ablation', values='macro_f1').loc[SEEDS]
    fam = [(o, (piv['all'] - piv[o]).mean(), stats.ttest_rel(piv['all'], piv[o])[1]) for o in ABL if o != 'all']
    for (o, dm, p), pa in zip(fam, holm([f[2] for f in fam])): ab_p.append(dict(split=sp, task=task, all_minus=o, mean_diff=dm, p_holm=pa))
AB_p = pd.DataFrame(ab_p); print('\nall-features minus ablation (positive = the removed group helped):'); print(AB_p.round(4).to_string(index=False))

## 9b. Feature engineering: which engineered features raise macro-F1?

Feature sets, all added to the 148 base features, XGBoost (default), 5 seeds, splits A and B, both tasks:
* `static`: angle sin/cos, per-phase V-I power-factor terms, log impedance, apparent-power and impedance-magnitude proxies, sequence-component ratios, and cross-relay differences/spreads of voltage, current, frequency and angle.
* `temporal_d1`: one-step difference only. `temporal_5_15`: difference + rolling mean/std over the past 5 and 15 rows. `temporal_15_30`: windows 15 and 30. Windows are strictly **causal** (past rows of the same run only), so they are usable in a streaming detector.
* `diag_runcontext` (**diagnostic, not deployable**): the same window features computed over a random permutation of the rows inside each run, so every window mixes past *and future* rows of the run. This measures how much plain run-level context (any rows of the same scenario) is worth. Two warnings: it is non-causal, and on split A it also mixes train and test rows of the same run, so **only split B is interpretable** for it. If it beats the causal sets, the gain from `temporal_*` is smoothing/context rather than anything specific to the order of events.

Causal rolling features are computed on the whole file and then split. They use only earlier rows of the same run (strictly past, label-free), so they are usable in a streaming detector; in split A the first 15-30 test rows of each run have windows that reach back into that run's training rows (features only, no labels).

In [ ]:
def static_feats():
    d = {}
    for r in RELAYS:
        for i in range(1, 13):
            pa = X[f'{r}-PA{i}:' + ('VH' if i in (1, 2, 3, 7, 8, 9) else 'IH')]; d[f'{r}_sin{i}'] = np.sin(np.deg2rad(pa)); d[f'{r}_cos{i}'] = np.cos(np.deg2rad(pa))
        for a_, b_ in ((1, 4), (2, 5), (3, 6)):
            ph = np.deg2rad(X[f'{r}-PA{a_}:VH'] - X[f'{r}-PA{b_}:IH']); d[f'{r}_pf{a_}'] = np.cos(ph); d[f'{r}_sinphi{a_}'] = np.sin(ph)
        d[f'{r}_logZ'] = np.log1p(X[f'{r}-PA:Z'])
        V = X[[f'{r}-PM{i}:V' for i in (1, 2, 3)]].values; I = X[[f'{r}-PM{i}:I' for i in (4, 5, 6)]].values
        d[f'{r}_S'] = V.mean(1) * I.mean(1); d[f'{r}_zmag'] = V.mean(1) / np.maximum(I.mean(1), 1e-6)
        d[f'{r}_seqV'] = X[f'{r}-PM8:V'] / np.maximum(X[f'{r}-PM7:V'], 1e-6); d[f'{r}_seqI'] = X[f'{r}-PM11:I'] / np.maximum(X[f'{r}-PM10:I'], 1e-6)
    vm = np.stack([X[[f'{r}-PM{i}:V' for i in (1, 2, 3)]].mean(1) for r in RELAYS], 1); im = np.stack([X[[f'{r}-PM{i}:I' for i in (4, 5, 6)]].mean(1) for r in RELAYS], 1)
    fm = np.stack([X[f'{r}:F'] for r in RELAYS], 1); an = np.stack([X[f'{r}-PA1:VH'] for r in RELAYS], 1)
    for nm, M in (('V', vm), ('I', im), ('F', fm)): d[f'{nm}_std'] = M.std(1); d[f'{nm}_range'] = np.ptp(M, 1)
    for a_ in range(4):
        for b_ in range(a_ + 1, 4):
            d[f'dV_{a_}{b_}'] = vm[:, a_] - vm[:, b_]; d[f'dI_{a_}{b_}'] = im[:, a_] - im[:, b_]; d[f'dAng_{a_}{b_}'] = np.sin(np.deg2rad(an[:, a_] - an[:, b_]))
    return pd.DataFrame(d, index=X.index)
SIG_COLS = DER_COLS + [f'{r}-PM{i}:V' for r in RELAYS for i in (1, 2, 3)] + [f'{r}-PM{i}:I' for r in RELAYS for i in (4, 5, 6)] + [f'{r}:F' for r in RELAYS]
def temporal_feats(windows, d1=True, order=None):
    S = X[SIG_COLS].copy(); gid = run_id
    if order is not None: S = S.iloc[order].reset_index(drop=True); gid = run_id          # rows permuted inside runs; run ids unchanged
    g = S.groupby(gid); parts = []
    if d1: parts.append((S - g.shift(1)).fillna(0).add_suffix('_d1'))
    for w in windows:
        parts.append(g.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True).sort_index().add_suffix(f'_m{w}'))
        parts.append(g.rolling(w, min_periods=2).std().reset_index(level=0, drop=True).sort_index().fillna(0).add_suffix(f'_s{w}'))
    out = pd.concat(parts, axis=1)
    if order is not None:                                                                    # restore original row alignment
        inv = np.empty_like(order); inv[order] = np.arange(len(order)); out = out.iloc[inv].reset_index(drop=True)
    return out
rs_ = np.random.RandomState(0); shuf = np.concatenate([np.flatnonzero(run_id == r_)[rs_.permutation(run_len[r_])] for r_ in range(n_runs)])
# shuf lists, run by run, the original row indices in a random order; S is reordered with shuf, so row k of S is original row shuf[k]
assert np.array_equal(run_id[shuf], np.sort(run_id))
ST = static_feats(); FE = {'base': X,
      'static': pd.concat([X, ST], axis=1),
      'temporal_d1': pd.concat([X, temporal_feats([], True)], axis=1),
      'temporal_5_15': pd.concat([X, temporal_feats([5, 15])], axis=1),
      'temporal_15_30': pd.concat([X, temporal_feats([15, 30])], axis=1),
      'diag_runcontext': pd.concat([X, temporal_feats([5, 15], order=shuf)], axis=1)}
print({k: v.shape[1] for k, v in FE.items()})

In [ ]:
fe_rows = []; t0 = time.time()
for k, df_ in FE.items():
    A_ = df_.values.astype('float32')
    for s in SEEDS:
        for sp, (a, b) in SPLITS.items():
            for task, y in TASKS.items():
                pr = fit_predict('xgb_default', A_[a], y[a], A_[b], s); fe_rows.append(dict(seed=s, split=sp, task=task, fset=k, **score(y[b], pr, task, marker[b])))
    print(k, f'done ({time.time()-t0:.0f}s)')
FER = pd.DataFrame(fe_rows)
FE_t = FER.groupby(['split', 'task', 'fset']).macro_f1.agg(['mean', 'std']).round(4).reset_index(); print(FE_t.to_string(index=False))
fe_p = []
for (sp, task), d in FER.groupby(['split', 'task']):
    piv = d.pivot(index='seed', columns='fset', values='macro_f1').loc[SEEDS]
    fam = [(o, (piv[o] - piv['base']).mean(), stats.ttest_rel(piv[o], piv['base'])[1]) for o in FE if o != 'base']
    for (o, dm, p), pa in zip(fam, holm([f[2] for f in fam])): fe_p.append(dict(split=sp, task=task, fset=o, gain_over_base=dm, p_holm=pa))
    dm = (piv['temporal_5_15'] - piv['diag_runcontext']).mean(); fe_p.append(dict(split=sp, task=task, fset='temporal_5_15 minus diag_runcontext', gain_over_base=dm, p_holm=stats.ttest_rel(piv['temporal_5_15'], piv['diag_runcontext'])[1]))
FE_p = pd.DataFrame(fe_p); print('\ngain over base (paired over seeds, Holm within family):'); print(FE_p.round(4).to_string(index=False))

## 10. Physical-Channel Reliance (PCR) on MSU

Same definition as on Sweeten: share of total class-averaged |SHAP| carried by the physical-channel features. On MSU the channels are the PMU measurements (raw + derived + Z-inf flags) against the 12 log columns. XGBoost default, 3-class, split A, exact TreeSHAP on up to 1,500 stratified test rows per seed; split B shown too.

In [ ]:
def shap_shares(a, b, seed):
    pr, m = fit_predict('xgb_default', Xv[a], g3[a], Xv[b], seed, return_model=True)
    rng = np.random.RandomState(seed); pick = np.concatenate([rng.choice(np.flatnonzero(g3[b] == c), min(SHAP_ROWS // 3, (g3[b] == c).sum()), replace=False) for c in range(3)])
    con = m.get_booster().predict(xgb.DMatrix(Xv[b][pick]), pred_contribs=True)[:, :, :-1]
    phi = np.abs(con).sum(1); out = {}
    for c in range(3):
        ph = phi[g3[b][pick] == c]; tot = ph.sum()
        out[G3_NAMES[c]] = {g: float(ph[:, [ix[f] for f in fs]].sum() / tot) for g, fs in GROUPS.items()}
    return out, phi.mean(0)
pcr_rows = []; phi_all = {}
for s in SEEDS:
    for sp, (a, b) in SPLITS.items():
        sh, phi = shap_shares(a, b, s); phi_all[(s, sp)] = phi
        for cls_, d in sh.items(): pcr_rows.append(dict(seed=s, split=sp, cls=cls_, pcr=1 - d['logs'], pcr_pmu_raw=d['pmu_raw'], derived=d['derived'], flags=d['flags'], logs=d['logs']))
PCR = pd.DataFrame(pcr_rows)
PCR_t = PCR.groupby(['split', 'cls'])[['pcr', 'pcr_pmu_raw', 'derived', 'flags', 'logs']].agg(lambda x: ms(x)).reset_index(); print(PCR_t.to_string(index=False))
top = pd.Series(np.mean([phi_all[(s, 'A_temporal')] for s in SEEDS], 0), index=FEATS).sort_values(ascending=False).head(15)
print('\ntop-15 features by mean |SHAP| (split A, mean over seeds):'); print(top.round(4).to_string())

## 11. Per-class difficulty and the natural-event / attack confusion (tuned XGBoost)

In [ ]:
cm = {sp: np.mean([CONF[(s, sp)] for s in SEEDS], 0) for sp in SPLITS}
for sp in SPLITS:
    rown = cm[sp] / cm[sp].sum(1, keepdims=True); print(f'\n3-class confusion, split {sp} (rows = truth, mean over seeds, row-normalised):'); print(pd.DataFrame(rown, index=G3_NAMES, columns=G3_NAMES).round(4).to_string())
pc = pd.DataFrame({sp: np.mean([PERCLS[(s, sp)] for s in SEEDS], 0) for sp in SPLITS}, index=[f'm{m}' for m in MARKERS])
pc['group'] = [G3_NAMES[g3[marker == m][0]] for m in MARKERS]
print('\n37-class per-marker F1, tuned XGBoost (mean over seeds); 10 hardest on split B:'); print(pc.sort_values('B_heldout_passes').head(10).round(4).to_string())
print('\nmean per-marker F1 by group:'); print(pc.groupby('group')[list(SPLITS)].mean().round(4).to_string())

## 12. Figure, tables, manifest

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
for j, task in enumerate(TASKS):
    for k, sp in enumerate(SPLITS):
        d = R[(R.task == task) & (R.split == sp)]; mu = d.groupby('model').macro_f1.mean().loc[MODELS]; sd = d.groupby('model').macro_f1.std().loc[MODELS]
        ax[k, j].bar(MODELS, mu, yerr=sd, capsize=3); ax[k, j].set_title(f'{task}, split {sp}'); ax[k, j].set_ylim(max(0, mu.min() - 0.1), 1.0); ax[k, j].set_ylabel('macro-F1')
plt.tight_layout(); plt.savefig(f'msu_models_{PIPELINE_VERSION}.png', dpi=150); plt.show()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for k, sp in enumerate(SPLITS):
    d = PCR[PCR.split == sp].groupby('cls')[['pcr_pmu_raw', 'derived', 'flags', 'logs']].mean().loc[G3_NAMES]; d.plot.bar(stacked=True, ax=ax[k]); ax[k].set_title(f'|SHAP| share by feature group, split {sp}'); ax[k].set_ylabel('share')
plt.tight_layout(); plt.savefig(f'msu_pcr_{PIPELINE_VERSION}.png', dpi=150); plt.show()
T = {'models': R, 'summary': summ, 'pairwise': PAIR, 'leakage': L.groupby(['task', 'split']).macro_f1.agg(['mean', 'std']).reset_index(), 'ablation': AB_t, 'ablation_tests': AB_p, 'feature_sets': FE_t, 'feature_set_tests': FE_p, 'pcr': PCR, 'per_marker_f1': pc.reset_index()}
for k_, t_ in T.items(): t_.to_csv(f'msu_{k_}_{PIPELINE_VERSION}.csv', index=False)
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'tables': T, 'best_params': BEST, 'confusion': CONF}, f'msu_results_{PIPELINE_VERSION}.pkl')
json.dump({'notebook': 'NB9', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'xgboost': xgb.__version__, 'rows': int(len(marker)), 'features': len(FEATS), 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb9_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=1)
for f in [f'msu_results_{PIPELINE_VERSION}.pkl', f'msu_models_{PIPELINE_VERSION}.png', f'msu_pcr_{PIPELINE_VERSION}.png', f'nb9_run_manifest_{PIPELINE_VERSION}.json', f'msu_xgb_tuned_params_{PIPELINE_VERSION}.json'] + [f'msu_{k_}_{PIPELINE_VERSION}.csv' for k_ in T]: print(('  OK  ' if os.path.exists(f) else 'MISSING '), f)